# Baseline Model Testing: YouTube Engagement Score Prediction (Regression)

This notebook implements baseline model testing for predicting continuous engagement scores on Algerian Baccalaureate educational videos.

## Objectives
1. Load engineered features
2. Create train/validation/test splits
3. Train multiple baseline regression models
4. Evaluate performance using regression metrics (MAE, RMSE, R²)
5. Analyze feature importance
6. **Save data splits to data/modeling/**

## Step 1: Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime
import warnings
import sys
import os
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
# Also set Python's built-in random if used
try:
    import random
    random.seed(RANDOM_SEED)
except ImportError:
    pass

# Fix path resolution - get project root
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path(__file__).parent if '__file__' in globals() else Path.cwd()
if NOTEBOOK_DIR.name != 'notebooks':
    # Try to find notebooks directory
    for parent in NOTEBOOK_DIR.parents:
        if (parent / 'notebooks').exists():
            NOTEBOOK_DIR = parent / 'notebooks'
            break
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / 'data'

# Add project root to path for imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📁 Data directory: {DATA_DIR}")
print(f"🎲 Random seed set to {RANDOM_SEED} for reproducibility")

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
# Note: Regression metrics (MAE, RMSE, R²) will be imported when needed in Step 7

# Ensemble model availability flags (optional - install xgboost and lightgbm)
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    print("⚠️ XGBoost not available. Install with: uv add xgboost")
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except ImportError:
    print("⚠️ LightGBM not available. Install with: uv add lightgbm")
    LIGHTGBM_AVAILABLE = False

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports complete")

## Step 2: Load Engineered Data

In [ ]:
# Load engineered features - try multiple paths
engineered_path = DATA_DIR / "processed" / "videos_engineered.csv"
bac_only_path = DATA_DIR / "processed" / "videos_bac_only.csv"

if engineered_path.exists():
    print(f"✅ Loading from: {engineered_path}")
    df = pd.read_csv(engineered_path)
elif bac_only_path.exists():
    print(f"⚠️ videos_engineered.csv not found. Loading {bac_only_path} and running feature engineering...")
    from src.features.engineer import VideoFeatureEngineer
    
    df_bac = pd.read_csv(bac_only_path)
    print(f"   Loaded {len(df_bac):,} Bac videos")
    
    # Run feature engineering
    engineer = VideoFeatureEngineer()
    df = engineer.fit_transform(df_bac)
    df = engineer.select_features(df, include_target=True)
    
    print(f"✅ Feature engineering complete: {df.shape[1]} features")
    
    # Save for future use
    engineered_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(engineered_path, index=False)
    print(f"💾 Saved to: {engineered_path}")
else:
    raise FileNotFoundError(
        f"❌ Neither {engineered_path} nor {bac_only_path} found.\n"
        f"   Run: uv run python run_pipeline.py filter_data --validate\n"
        f"   Then: uv run python -m src.features.build_features --input {DATA_DIR / 'processed' / 'videos_bac_only.csv'}"
    )

# Verify required columns
required_cols = ['engagement_score', 'channel_id']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"❌ Missing required columns: {missing_cols}")

print(f"\n📊 Loaded {len(df):,} videos")
print(f"Features: {df.shape[1]} columns")
print(f"\nEngagement score statistics:")
print(df['engagement_score'].describe())
if 'subject' in df.columns:
    print(f"\nSubject distribution:")
    print(df['subject'].value_counts())

# Display first few rows
df.head()

# Step 2.5: Data Cleaning (Handle Missing Values)


In [ ]:

# 1. Add transcript availability indicator
transcript_cols = ['transcript_word_count', 'transcript_char_count', 'transcript_sentence_count']
df['has_transcript'] = df[transcript_cols[0]].notna().astype(int)
print(f"📊 Videos with transcripts: {df['has_transcript'].sum():,} ({df['has_transcript'].mean()*100:.1f}%)")

# 2. Fill transcript-derived features with 0 (no transcript = no data)
transcript_feature_cols = [col for col in df.columns if any(x in col.lower() for x in 
    ['transcript', 'flesch', 'fog', 'readability', 'speech_rate', 'lexical', 
     'sentence', 'question', 'example', 'explanation', 'contrast', 'keyword_count'])]
for col in transcript_feature_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# 3. Fill description NaN
df['description'] = df['description'].fillna('')
df['description_length'] = df['description'].str.len()

# 4. Report cleaning results
print(f"✅ Filled NaN in {len(transcript_feature_cols)} transcript columns with 0")
print(f"✅ Remaining NaN columns: {df.isnull().sum().sum()}")

# Step 2.6: Final Imputation and Data Export 
Based on Data_cleaning_guide.md recommendations

In [ ]:


print("🔧 Final Imputation Pass")
print("=" * 60)

# 1. Text fields -> empty string
df['description'] = df['description'].fillna('')
df['tags'] = df['tags'].fillna('') if 'tags' in df.columns else ''

# 2. Transcript-derived numerical features -> 0
# (For videos without transcripts, these should be 0, not NaN)
transcript_numeric_cols = [
    'transcript_word_count', 'transcript_char_count', 'transcript_sentence_count',
    'unique_word_count', 'lexical_diversity', 'avg_words_per_sentence',
    'flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog_index',
    'automated_readability_index', 'speech_rate_wpm', 'speech_rate_optimal',
    'question_count', 'question_density', 'example_count', 'example_density',
    'explanation_count', 'explanation_density', 'contrast_count', 'contrast_density',
    'technical_term_count', 'technical_term_density', 
    'subject_keyword_count', 'subject_keyword_density'
]

for col in transcript_numeric_cols:
    if col in df.columns:
        missing = df[col].isna().sum()
        if missing > 0:
            df[col] = df[col].fillna(0)
            print(f"  ✓ {col}: {missing:,} NaN → 0")

# 3. Verify no remaining NaN in key columns
remaining_nan = df.isnull().sum()
cols_with_nan = remaining_nan[remaining_nan > 0]
if len(cols_with_nan) > 0:
    print(f"\n⚠️ Remaining NaN columns:")
    print(cols_with_nan.to_string())
else:
    print(f"\n✅ No remaining NaN values!")

print(f"\n📊 Final dataset shape: {df.shape}")

# Step 2.7: Save Final Imputed Dataset


In [ ]:
# ============================================================
# Step 2.7: Save Final Imputed Dataset
# ============================================================

output_path = DATA_DIR / "cleaned" / "data_final.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save the fully imputed dataset
df.to_csv(output_path, index=False)

print(f"💾 Saved final imputed dataset to: {output_path}")
print(f"   Shape: {df.shape}")
print(f"   Size: {output_path.stat().st_size / (1024*1024):.2f} MB")

# Verification
df_check = pd.read_csv(output_path, nrows=5)
print(f"   ✓ File readable, columns: {len(df_check.columns)}")

## Step 3: Feature Preparation

In [ ]:
# Define target and features
target_col = 'engagement_score'

# Exclude columns:
# 1. ID columns (not features)
# 2. Target variable
# 3. Raw engagement counts (view_count, like_count, comment_count) - DATA LEAKAGE!
#    These are used to compute engagement_score and engagement_category
# 4. Derived engagement metrics that use raw counts (like_ratio, comment_ratio, engagement_score)
# 5. Raw text columns (title, channel_title, description, tags, publish_date)
#    We only use extracted numerical features from text (title_length, description_length, etc.)
# 6. publish_day_of_week (string) - we have is_weekday binary feature instead
exclude_cols = [
    # IDs
    'video_id', 
    'channel_id',
    'channel_title',
    # Target
    target_col,
    # Classification target (we use regression now)
    'engagement_category',
    # Raw engagement counts (CAUSES DATA LEAKAGE!)
    'view_count',
    'like_count', 
    'comment_count',
    # Derived engagement metrics (also used in target)
    'engagement_score',
    'like_ratio',
    'comment_ratio',
    # Channel engagement (derived from engagement_score - potential leakage)
    'channel_avg_engagement',
    # Raw text columns (use only extracted features)
    'title',      # Exclude raw text - use TF-IDF features instead
    'description',  # Exclude raw text - use TF-IDF features instead
    'tags',
    'publish_date',  # Use only extracted temporal features (days_since_publish, publish_hour, etc.)
    'publish_day_of_week',  # String column - we have is_weekday binary instead
    # removed to prevent leakage (test if the model relies more on specific channel stats)
    "channel_video_count",
    "channel_avg_views",
    "channel_video_avg_duration",
    "channel_age_days"
    # "subject",
]

# Get feature columns (exclude the above)
feature_cols = [col for col in df.columns if col not in exclude_cols]

# Identify categorical features for encoding (subject is the main one)
# Only include object columns that make sense as categorical features
categorical_features = []
for col in feature_cols:
    if df[col].dtype == 'object':
        categorical_features.append(col)

numerical_features = [col for col in feature_cols if col not in categorical_features]

# Verify all numerical features are actually numeric
for col in numerical_features:
    if not pd.api.types.is_numeric_dtype(df[col]):
        print(f"⚠️ Warning: {col} is not numeric but in numerical_features")

print(f"🎯 Target: {target_col}")
print(f"📊 Total features: {len(feature_cols)}")
print(f"   - Numerical: {len(numerical_features)}")
print(f"   - Categorical: {len(categorical_features)}")

print(f"\n✅ Included features:")
print(f"\nTemporal features:")
temporal = [col for col in numerical_features if any(x in col for x in ['days_since', 'publish_hour', 'publish_month', 'publish_year', 'is_evening', 'is_weekday'])]
print(temporal)

print(f"\nText extraction features (no raw text):")
text_extracted = [col for col in numerical_features if any(x in col for x in ['length', 'word_count', 'tag_count', 'is_exam'])]
print(text_extracted)

print(f"\nChannel features:")
channel_features = [col for col in numerical_features if 'channel' in col]
print(channel_features)

print(f"\nContent features:")
content_features = [col for col in numerical_features if col in ['duration_sec']]
print(content_features)

print(f"\nCategorical features (will be one-hot encoded):")
print(categorical_features)

print(f"\n❌ Excluded (to prevent data leakage):")

excluded_present = [col for col in exclude_cols if col in df.columns]
print(excluded_present)

In [ ]:
# One-hot encode categorical features
# IMPORTANT: Use sorted categories to ensure consistent column ordering for reproducibility
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

# Reorder all columns alphabetically for reproducibility (ensures consistent feature matrix)
# This prevents feature shuffling when data is loaded in different order
df_encoded = df_encoded.reindex(sorted(df_encoded.columns), axis=1)

# Update feature columns after encoding - use SAME exclude_cols to ensure consistency
# This ensures text columns, raw counts, and derived engagement metrics are excluded
encoded_feature_cols = [col for col in df_encoded.columns if col not in exclude_cols]
# Sort feature columns alphabetically for reproducibility
encoded_feature_cols = sorted(encoded_feature_cols)

# Verify all encoded features are numeric (after one-hot encoding)
non_numeric = [col for col in encoded_feature_cols if not pd.api.types.is_numeric_dtype(df_encoded[col])]
if non_numeric:
    print(f"⚠️ Warning: Non-numeric columns found: {non_numeric}")
    print("   These will be excluded from feature set.")
    encoded_feature_cols = [col for col in encoded_feature_cols if pd.api.types.is_numeric_dtype(df_encoded[col])]
    encoded_feature_cols = sorted(encoded_feature_cols)  # Re-sort after filtering

print(f"✅ After encoding: {len(encoded_feature_cols)} features")
print(f"\nNew subject dummy columns:")
subject_dummies = [col for col in encoded_feature_cols if col.startswith('subject_')]
print(subject_dummies)

print(f"\n✅ All features are numeric (ready for scaling)")
print(f"   Sample features: {encoded_feature_cols[:5]}...")

## Step 4: Create Train/Val/Test Splits (Within-Channel Splitting)

We split videos randomly across all channels (60% train, 20% val, 20% test).
Since we include channel-level features explicitly (channel_video_count, channel_avg_views, etc.), 
splitting within channels is acceptable for regression.

In [ ]:
# Split videos WITHIN each channel (not by channel)
# This ensures balanced splits: 60% train, 20% val, 20% test
# Since we include channel-level features explicitly, this is acceptable

print(f"📺 Total unique channels: {df_encoded['channel_id'].nunique()}")
print(f"📹 Total videos: {len(df_encoded):,}")

# Strategy: Split videos randomly across all channels
# No stratification needed for regression

# First split: Train+Val (80%) vs Test (20%)
train_val_idx, test_idx = train_test_split(
    df_encoded.index,
    test_size=0.20,
    random_state=RANDOM_SEED
)

# Second split: Train (75% of train_val) vs Val (25% of train_val)
# Final: Train 60%, Val 20%, Test 20%
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.25,  # 0.25 * 0.80 = 0.20 of total
    random_state=RANDOM_SEED
)

# Create split datasets
train_df = df_encoded.loc[train_idx].copy()
val_df = df_encoded.loc[val_idx].copy()
test_df = df_encoded.loc[test_idx].copy()

print(f"\n✅ Split complete (videos split within channels):")
print(f"Train: {len(train_df):,} videos ({len(train_df)/len(df_encoded)*100:.1f}%)")
print(f"Val: {len(val_df):,} videos ({len(val_df)/len(df_encoded)*100:.1f}%)")
print(f"Test: {len(test_df):,} videos ({len(test_df)/len(df_encoded)*100:.1f}%)")

# Show channel distribution across splits
print(f"\n📺 Channels represented in each split:")
print(f"Train: {train_df['channel_id'].nunique()} unique channels")
print(f"Val: {val_df['channel_id'].nunique()} unique channels")
print(f"Test: {test_df['channel_id'].nunique()} unique channels")

# Note about overlap (expected - same channels appear in all splits)
print(f"\n⚠️ Note: Videos from the same channel appear in different splits.")
print(f"   This is intentional - we're testing within-channel prediction.")

In [ ]:
# Splits already created above - verify and show statistics

# Check engagement score distribution across splits
print("\n📊 Engagement score distribution across splits:")
for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n{split_name}:")
    print(f"  Mean: {split_df['engagement_score'].mean():.4f}")
    print(f"  Std: {split_df['engagement_score'].std():.4f}")
    print(f"  Min: {split_df['engagement_score'].min():.4f}")
    print(f"  Max: {split_df['engagement_score'].max():.4f}")

# Verify all channels are represented (some overlap is expected)
all_channels_train = set(train_df['channel_id'].unique())
all_channels_val = set(val_df['channel_id'].unique())
all_channels_test = set(test_df['channel_id'].unique())

channels_in_all_splits = all_channels_train & all_channels_val & all_channels_test
unique_channels = sorted(df_encoded['channel_id'].unique())  # For compatibility with later cells
print(f"\n✅ Channels appearing in all splits: {len(channels_in_all_splits)}/{len(unique_channels)}")
print(f"   This ensures we're testing within-channel generalization.")

## Step 5: Save Data Splits to data/modeling/

**Critical**: Save train/val/test splits for reproducibility and future use

In [ ]:
# Create modeling directory
modeling_dir = DATA_DIR / "modeling"
modeling_dir.mkdir(parents=True, exist_ok=True)

# Save splits as CSV
train_df.to_csv(modeling_dir / "train.csv", index=False)
val_df.to_csv(modeling_dir / "val.csv", index=False)
test_df.to_csv(modeling_dir / "test.csv", index=False)

print("💾 Saved data splits:")
print(f"  - {modeling_dir / 'train.csv'}")
print(f"  - {modeling_dir / 'val.csv'}")
print(f"  - {modeling_dir / 'test.csv'}")

# Save split mapping (video_id to split assignment)
split_mapping = pd.DataFrame({
    'video_id': list(train_df['video_id']) + list(val_df['video_id']) + list(test_df['video_id']),
    'split': ['train'] * len(train_df) + ['val'] * len(val_df) + ['test'] * len(test_df)
})
split_mapping.to_csv(modeling_dir / "splits_mapping.csv", index=False)
print(f"  - {modeling_dir / 'splits_mapping.csv'}")

# Save metadata
metadata = {
    'created_at': datetime.now().isoformat(),
    'total_videos': len(df_encoded),
    'total_channels': len(unique_channels),
    'train_videos': len(train_df),
    'val_videos': len(val_df),
    'test_videos': len(test_df),
    'train_channels': train_df['channel_id'].nunique(),
    'val_channels': val_df['channel_id'].nunique(),
    'test_channels': test_df['channel_id'].nunique(),
    'features': encoded_feature_cols,
    'target': target_col,
    'random_state': 42
}

with open(modeling_dir / "split_metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"  - {modeling_dir / 'split_metadata.json'}")

print("\n✅ All splits saved successfully!")

## Step 6: Prepare X and y for Modeling

In [ ]:
# Prepare features and target
# IMPORTANT: Use sorted encoded_feature_cols to ensure consistent column ordering
# This prevents feature matrix shuffling when notebook is rerun
X_train = train_df[encoded_feature_cols].values  # encoded_feature_cols is already sorted
y_train = train_df[target_col].values

X_val = val_df[encoded_feature_cols].values
y_val = val_df[target_col].values

X_test = test_df[encoded_feature_cols].values
y_test = test_df[target_col].values

# No label encoding needed for regression - target is already numeric

print(f"\n✅ Prepared datasets:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")
print(f"\nTarget statistics:")
print(f"Train: Mean={y_train.mean():.4f}, Std={y_train.std():.4f}")
print(f"Val: Mean={y_val.mean():.4f}, Std={y_val.std():.4f}")
print(f"Test: Mean={y_test.mean():.4f}, Std={y_test.std():.4f}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled using StandardScaler")

In [ ]:
# ============================================================
# ARABERT SEMANTIC FEATURE EXTRACTION - OPTIMIZED VERSION
# Uses caching to avoid recomputing embeddings
# ============================================================

from src.features.text_embeddings import TextEmbeddingExtractor
import numpy as np
from pathlib import Path
import pickle
import torch
from sklearn.decomposition import PCA

# Check if embeddings are already cached
cache_dir = DATA_DIR / 'modeling'
cache_path = cache_dir / 'arabert_embeddings.pkl'

if cache_path.exists():
    print("📦 Loading cached AraBERT embeddings...")
    with open(cache_path, 'rb') as f:
        cached = pickle.load(f)
        X_train_bert = cached['X_train_bert']
        X_val_bert = cached['X_val_bert']
        X_test_bert = cached['X_test_bert']
    print(f"✅ Loaded cached embeddings: {X_train_bert.shape[1]} dimensions")
else:
    print("📝 Extracting semantic embeddings using AraBERT v2...")
    print("   (pre-trained on Modern Standard Arabic - better for formal educational titles)")
    
    # Check if GPU is available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("   ⚠️ Running on CPU - this will take ~30-45 minutes")
        print("   💡 If you have a GPU, ensure CUDA is installed for 10x speedup\n")
    else:
        print(f"   🚀 Running on GPU ({torch.cuda.get_device_name(0)}) - should take ~2-5 minutes\n")
    
    # IMPORTANT: train_df, val_df, test_df were created from df_encoded (encoded features only)
    # We need to get 'title' and 'description' from the original df DataFrame
    # Create text DataFrames from original df using the same indices
    train_df_text = df.loc[train_idx, ['title', 'description']].copy()
    val_df_text = df.loc[val_idx, ['title', 'description']].copy()
    test_df_text = df.loc[test_idx, ['title', 'description']].copy()
    
    # Combine title + description for richer context
    train_df_text['text_combined'] = (
        train_df_text['title'].fillna('') + ' ' + 
        train_df_text['description'].fillna('')
    )
    val_df_text['text_combined'] = (
        val_df_text['title'].fillna('') + ' ' + 
        val_df_text['description'].fillna('')
    )
    test_df_text['text_combined'] = (
        test_df_text['title'].fillna('') + ' ' + 
        test_df_text['description'].fillna('')
    )
    
    # Load AraBERT model (will download on first run: ~540MB)
    # Defaults to "aubmindlab/bert-base-arabertv2"
    embedding_extractor = TextEmbeddingExtractor(device=device)
    
    # Use smaller batch size for CPU, larger for GPU
    batch_size = 64 if device == "cuda" else 16
    
    # Extract embeddings from training data
    print("  Processing training set...")
    X_train_bert = embedding_extractor.get_embeddings(
        train_df_text['text_combined'].tolist(),
        batch_size=batch_size
    )
    
    # Extract embeddings from validation data
    print("  Processing validation set...")
    X_val_bert = embedding_extractor.get_embeddings(
        val_df_text['text_combined'].tolist(),
        batch_size=batch_size
    )
    
    # Extract embeddings from test data
    print("  Processing test set...")
    X_test_bert = embedding_extractor.get_embeddings(
        test_df_text['text_combined'].tolist(),
        batch_size=batch_size
    )
    
    # Cache the embeddings for future runs
    print("\n💾 Caching embeddings for future use...")
    with open(cache_path, 'wb') as f:
        pickle.dump({
            'X_train_bert': X_train_bert,
            'X_val_bert': X_val_bert,
            'X_test_bert': X_test_bert
        }, f)
    print(f"   Saved to: {cache_path}")

# ============================================================
# Dimensionality Reduction: Reduce AraBERT (768) → 100 using PCA
# ============================================================
print("\n🔻 Reducing AraBERT embeddings dimensionality using PCA (768 → 100)...")
pca = PCA(n_components=100, random_state=42)
X_train_bert_reduced = pca.fit_transform(X_train_bert)
X_val_bert_reduced = pca.transform(X_val_bert)
X_test_bert_reduced = pca.transform(X_test_bert)
print(f"AraBERT reduced: {X_train_bert.shape[1]} → {X_train_bert_reduced.shape[1]} features")
print(f"Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

# Combine numerical features with reduced semantic embeddings
X_train_with_text = np.hstack([X_train_scaled, X_train_bert_reduced])
X_val_with_text = np.hstack([X_val_scaled, X_val_bert_reduced])
X_test_with_text = np.hstack([X_test_scaled, X_test_bert_reduced])

print(f"\n✅ Semantic embeddings extracted and reduced (AraBERT):")
print(f"   Original numerical features: {X_train_scaled.shape[1]}")
print(f"   AraBERT (reduced) semantic features: {X_train_bert_reduced.shape[1]}")
print(f"   Total combined features: {X_train_with_text.shape[1]}")
print(f"\n💡 Note: AraBERT captures semantic meaning in MSA (Modern Standard Arabic).")
print(f"   Better suited for educational video titles than dialect models.")


## Step 6.5: Extract TF-IDF Features from Text

Add semantic features from video titles and descriptions using TF-IDF (Term Frequency-Inverse Document Frequency).
This captures topic-specific keywords that distinguish high-engagement videos from low-engagement ones.

In [ ]:
# ============================================================
# STEP 6.5: EXTRACT TF-IDF FEATURES FROM TEXT
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

print("📝 Extracting TF-IDF features from video titles and descriptions...")

# Prepare text data (same as used for AraBERT)
# FIX: previous code referenced train_df_text, val_df_text, test_df_text which may not be defined here.
# Let's use the same DataFrames already used for embedding extraction, which are train_df, val_df, test_df.
# Create text DataFrames from original df using the same indices
train_df_text = df.loc[train_idx, ['title', 'description']].copy()
val_df_text = df.loc[val_idx, ['title', 'description']].copy()
test_df_text = df.loc[test_idx, ['title', 'description']].copy()

# Combine title + description for richer context
train_df_text['text_combined'] = (
    train_df_text['title'].fillna('') + ' ' + 
    train_df_text['description'].fillna('')
)
val_df_text['text_combined'] = (
    val_df_text['title'].fillna('') + ' ' + 
    val_df_text['description'].fillna('')
)
test_df_text['text_combined'] = (
    test_df_text['title'].fillna('') + ' ' + 
    test_df_text['description'].fillna('')
)
train_text_combined = train_df_text['text_combined'].tolist()
val_text_combined = val_df_text['text_combined'].tolist()
test_text_combined = test_df_text['text_combined'].tolist()

# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=100,      # Top 1500 most important terms
    min_df=2,               # Term must appear in at least 2 documents
    max_df=0.8,             # Ignore terms that appear in >80% of documents
    ngram_range=(1, 2),     # Unigrams and bigrams
    sublinear_tf=True       # Use log scaling for term frequency
)

# Fit on training data and transform all sets
print("  Processing training set...")
X_train_tfidf = tfidf.fit_transform(train_text_combined)

print("  Processing validation set...")
X_val_tfidf = tfidf.transform(val_text_combined)

print("  Processing test set...")
X_test_tfidf = tfidf.transform(test_text_combined)

# Convert sparse matrices to dense arrays for combining with other features
X_train_tfidf = X_train_tfidf.toarray()
X_val_tfidf = X_val_tfidf.toarray()
X_test_tfidf = X_test_tfidf.toarray()

print(f"\n✅ TF-IDF features extracted:")
print(f"   Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"   Feature dimensions: {X_train_tfidf.shape[1]}")
print(f"   Sample terms: {list(tfidf.get_feature_names_out()[:10])}")

## Step 6.6: Combine All Features (Strategy 1)

**Improvement Strategy**: Combine TF-IDF + AraBERT + Numerical Features

### Why This Works

Based on research (documented in `docs/02_model_baseline_testing/arabert_implementation_guide.md`):

1. **TF-IDF captures discriminative keywords**: "بكالوريا", "رياضيات", specific subject terms
2. **AraBERT captures semantic meaning**: Understands context and relationships between concepts
3. **Numerical features capture engagement patterns**: Views, duration, like ratios, etc.

### Expected Results

| Configuration | Expected R² | Expected MAE |
|--------------|-------------|--------------|
| AraBERT only | 0.34-0.43 | 0.69-0.72 |
| TF-IDF only | 0.44-0.46 | 0.62-0.64 |
| **Combined (All 3)** | **0.48-0.52** | **0.58-0.62** |

The combination leverages the strengths of each feature type, leading to better overall performance than any single approach.

In [ ]:
# ============================================================
# STRATEGY 1: COMBINE ALL FEATURES (TF-IDF + AraBERT + NUMERICAL)
# Implementing improvement strategy from arabert_implementation_guide.md
# ============================================================

print("🔗 Combining TF-IDF + AraBERT + Numerical features...")
print("   This combines the strengths of:")
print("   - TF-IDF: Discriminative keywords (task-specific)")
print("   - AraBERT: Semantic understanding (context-aware)")
print("   - Numerical: Engagement metrics (views, duration, etc.)\n")

# Combine all three feature types
X_train_combined = np.hstack([
    X_train_scaled,      # 14 numerical features
    X_train_tfidf,       # 1500 TF-IDF features
    X_train_bert         # 768 AraBERT features
])

X_val_combined = np.hstack([
    X_val_scaled,
    X_val_tfidf,
    X_val_bert
])

X_test_combined = np.hstack([
    X_test_scaled,
    X_test_tfidf,
    X_test_bert
])

print(f"✅ Feature combination complete:")
print(f"   Numerical features:  {X_train_scaled.shape[1]:>4} dims")
print(f"   TF-IDF features:     {X_train_tfidf.shape[1]:>4} dims")
print(f"   AraBERT features:    {X_train_bert.shape[1]:>4} dims")
print(f"   {'─' * 35}")
print(f"   Total combined:      {X_train_combined.shape[1]:>4} dims")
print(f"\n💡 Expected improvement: R² 0.48-0.52 (vs 0.43 AraBERT-only)")

# Update X_train_with_text to use combined features for model training
X_train_with_text = X_train_combined
X_val_with_text = X_val_combined
X_test_with_text = X_test_combined

In [ ]:
# Define regression models
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

models = {
    'Ridge Regression': Ridge(alpha=1.0, random_state=42),
    'Lasso Regression': Lasso(alpha=1.0, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_depth=20),
    # 'SVR': SVR(kernel='rbf'),  # SKIPPED: Extremely slow on large datasets, poor performance
    #'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=5)
}

# Add ensemble models if available (with early stopping for faster training)
if XGBOOST_AVAILABLE:
    from xgboost import XGBRegressor
    models['XGBoost'] = XGBRegressor(random_state=42, n_estimators=100, early_stopping_rounds=10, max_depth=5, device='cuda')
    
if LIGHTGBM_AVAILABLE:
    from lightgbm import LGBMRegressor
    models['LightGBM'] = LGBMRegressor(random_state=42, verbose=-1, n_estimators=100, max_depth=5, n_jobs=-1) 

print(f"🤖 Training {len(models)} baseline models:")
for model_name in models.keys():
    print(f"  - {model_name}")

In [ ]:
# Train and evaluate models
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = {}

for model_name, model in models.items():
    print(f"\n🔧 Training {model_name}...")
    
    try:
        # PERFORMANCE: Use sparse matrices for Ridge/Lasso (much faster)
        if model_name in ['Ridge Regression', 'Lasso Regression']:
            model.fit(X_train_with_text, y_train)
            y_val_pred = model.predict(X_val_with_text)
        # Early stopping for XGBoost
        elif model_name == 'XGBoost' and XGBOOST_AVAILABLE:
            model.fit(
                X_train_with_text, y_train,
                eval_set=[(X_val_with_text, y_val)],
                verbose=False
            )
            y_val_pred = model.predict(X_val_with_text)
        # Early stopping for LightGBM
        elif model_name == 'LightGBM' and LIGHTGBM_AVAILABLE:
            model.fit(
                X_train_with_text, y_train,
                eval_set=[(X_val_with_text, y_val)]
            )
            y_val_pred = model.predict(X_val_with_text)
        else:
            # Train with TF-IDF features (dense arrays for other models)
            model.fit(X_train_with_text, y_train)
            y_val_pred = model.predict(X_val_with_text)
        
        # Calculate regression metrics
        mae = mean_absolute_error(y_val, y_val_pred)
        mse = mean_squared_error(y_val, y_val_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_val, y_val_pred)
        
        results[model_name] = {
            'model': model,
            'mae': mae,
            'mse': mse,
            'rmse': rmse,
            'r2_score': r2,
            'predictions': y_val_pred
        }
        
        print(f"  ✅ MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}")
        
    except Exception as e:
        print(f"  ❌ Error: {e}")
        continue

print("\n✅ All models trained!")

## Step 8: Model Performance Comparison

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'MAE': [results[m]['mae'] for m in results.keys()],
    'RMSE': [results[m]['rmse'] for m in results.keys()],
    'R²': [results[m]['r2_score'] for m in results.keys()]
})

# Sort by R² (higher is better)
comparison_df = comparison_df.sort_values('R²', ascending=False)
print("📊 Model Performance on Validation Set:")
print(comparison_df.to_string(index=False))

# Find best model (highest R²)
best_model_name = comparison_df.iloc[0]['Model']
print(f"\n🏆 Best Model: {best_model_name}")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x = np.arange(len(comparison_df))

# MAE (lower is better)
axes[0].bar(x, comparison_df['MAE'], color='coral', alpha=0.8)
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Mean Absolute Error')
axes[0].set_title('MAE (Lower is Better)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# RMSE (lower is better)
axes[1].bar(x, comparison_df['RMSE'], color='steelblue', alpha=0.8)
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Root Mean Squared Error')
axes[1].set_title('RMSE (Lower is Better)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

# R² (higher is better)
axes[2].bar(x, comparison_df['R²'], color='seagreen', alpha=0.8)
axes[2].set_xlabel('Model')
axes[2].set_ylabel('R² Score')
axes[2].set_title('R² Score (Higher is Better)')
axes[2].set_xticks(x)
axes[2].set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
axes[2].grid(axis='y', alpha=0.3)
axes[2].axhline(y=0, color='red', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()

## Step 9: Detailed Analysis of Best Model

In [ ]:
# Get best model
best_model = results[best_model_name]['model']
y_val_pred = results[best_model_name]['predictions']

# Regression analysis
print(f"📊 Detailed Metrics for {best_model_name}:")
print(f"  MAE:  {results[best_model_name]['mae']:.4f}")
print(f"  MSE:  {results[best_model_name]['mse']:.4f}")
print(f"  RMSE: {results[best_model_name]['rmse']:.4f}")
print(f"  R²:   {results[best_model_name]['r2_score']:.4f}")


# Error analysis
residuals = y_val - y_val_pred
print(f"\n📈 Error Distribution:")
print(f"  Mean residual: {residuals.mean():.4f}")
print(f"  Std residual:  {residuals.std():.4f}")
print(f"  Min residual:  {residuals.min():.4f}")
print(f"  Max residual:  {residuals.max():.4f}")

In [ ]:
# Scatter plot: Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Predicted vs Actual
axes[0].scatter(y_val, y_val_pred, alpha=0.5, s=20)
axes[0].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Engagement Score')
axes[0].set_ylabel('Predicted Engagement Score')
axes[0].set_title(f'Predicted vs Actual - {best_model_name}')
axes[0].legend()
axes[0].grid(alpha=0.3)


# Right: Residual plot

residuals = y_val - y_val_pred

axes[1].scatter(y_val_pred, residuals, alpha=0.5, s=20)

axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].grid(alpha=0.3)

axes[1].set_xlabel('Predicted Engagement Score')
axes[1].set_ylabel('Residuals (Actual - Predicted)')
axes[1].set_title(f'Residual Plot - {best_model_name}')

plt.tight_layout()
plt.show()
axes[1].set_ylabel('Residuals (Actual - Predicted)')

## Step 10: Feature Importance Analysis

In [ ]:
# Extract feature importance (if available)
# Check if TF-IDF features were used (model was trained with X_train_with_text)
try:
    # Check if TF-IDF features exist
    if 'tfidf' in globals() and 'X_train_with_text' in globals():
        # Model includes TF-IDF features - combine feature names
        numerical_feature_names = encoded_feature_cols
        tfidf_feature_names = [f"tfidf_{term}" for term in tfidf.get_feature_names_out()]
        all_feature_names = numerical_feature_names + tfidf_feature_names
    else:
        # No TF-IDF features - use only numerical features
        all_feature_names = encoded_feature_cols
except NameError:
    # TF-IDF not defined - use only numerical features
    all_feature_names = encoded_feature_cols

if hasattr(best_model, 'feature_importances_'):
    # Tree-based models
    importances = best_model.feature_importances_
    
    # Verify feature count matches
    if len(importances) != len(all_feature_names):
        print(f"⚠️ Warning: Feature count mismatch!")
        print(f"   Model has {len(importances)} features")
        print(f"   Feature names list has {len(all_feature_names)} features")
        print(f"   Using first {min(len(importances), len(all_feature_names))} features")
        min_len = min(len(importances), len(all_feature_names))
        importances = importances[:min_len]
        all_feature_names = all_feature_names[:min_len]
    
    feature_importance_df = pd.DataFrame({
        'feature': all_feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print(f"📊 Top 15 Most Important Features ({best_model_name}):")
    print(feature_importance_df.head(15).to_string(index=False))
    
    # Plot
    plt.figure(figsize=(10, 8))
    top_n = 20
    top_features = feature_importance_df.head(top_n)
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Importance')
    plt.title(f'Top {top_n} Feature Importances - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
elif hasattr(best_model, 'coef_'):
    # Linear models
    importances = np.abs(best_model.coef_).mean(axis=0)
    
    # Verify feature count matches
    if len(importances) != len(all_feature_names):
        print(f"⚠️ Warning: Feature count mismatch!")
        print(f"   Model has {len(importances)} features")
        print(f"   Feature names list has {len(all_feature_names)} features")
        print(f"   Using first {min(len(importances), len(all_feature_names))} features")
        min_len = min(len(importances), len(all_feature_names))
        importances = importances[:min_len]
        all_feature_names = all_feature_names[:min_len]
    
    feature_importance_df = pd.DataFrame({
        'feature': all_feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print(f"📊 Top 15 Most Important Features ({best_model_name}):")
    print(feature_importance_df.head(15).to_string(index=False))
    
    # Plot
    plt.figure(figsize=(10, 8))
    top_n = 20
    top_features = feature_importance_df.head(top_n)
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Absolute Coefficient Value')
    plt.title(f'Top {top_n} Feature Importances - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
else:
    print(f"⚠️ {best_model_name} does not support feature importance extraction")

In [ ]:
# Analyze TF-IDF Feature Contribution
if hasattr(best_model, 'feature_importances_'):
    # Get all feature names (numerical + TF-IDF)
    numerical_feature_names = encoded_feature_cols
    tfidf_feature_names = [f"tfidf_{term}" for term in tfidf.get_feature_names_out()]
    all_feature_names = numerical_feature_names + tfidf_feature_names

    # Get importances
    importances = best_model.feature_importances_
    
    # --- Fix: Ensure all arrays are the same length ---
    len_names = len(all_feature_names)
    len_importances = len(importances)
    len_types = len(numerical_feature_names) + len(tfidf_feature_names)
    min_len = min(len_names, len_importances, len_types)
    if len_names != len_importances or len_names != len_types:
        print(f"⚠️ Array length mismatch in feature importance analysis:")
        print(f"   - all_feature_names: {len_names}")
        print(f"   - importances: {len_importances}")
        print(f"   - types: {len_types}")
        print(f"   => Truncating all to {min_len} elements for consistency.")
    all_feature_names = all_feature_names[:min_len]
    importances = importances[:min_len]
    type_labels = (['numerical']*len(numerical_feature_names) + ['tfidf']*len(tfidf_feature_names))[:min_len]

    # Create dataframe
    feature_df = pd.DataFrame({
        'feature': all_feature_names,
        'importance': importances,
        'type': type_labels
    }).sort_values('importance', ascending=False)

    # Show top TF-IDF features
    top_tfidf = feature_df[feature_df['type'] == 'tfidf'].head(15)
    print("📊 Top 15 TF-IDF Features by Importance:")
    print(top_tfidf[['feature', 'importance']].to_string(index=False))

    # Compare contribution: numerical vs text
    numerical_contrib = feature_df[feature_df['type'] == 'numerical']['importance'].sum()
    tfidf_contrib = feature_df[feature_df['type'] == 'tfidf']['importance'].sum()

    print(f"\n📈 Feature Type Contribution:")
    if numerical_contrib + tfidf_contrib > 0:
        print(f"   Numerical features: {numerical_contrib:.3f} ({numerical_contrib/(numerical_contrib+tfidf_contrib)*100:.1f}%)")
        print(f"   TF-IDF features: {tfidf_contrib:.3f} ({tfidf_contrib/(numerical_contrib+tfidf_contrib)*100:.1f}%)")
    else:
        print(f"   Numerical features: {numerical_contrib:.3f} (0.0%)")
        print(f"   TF-IDF features: {tfidf_contrib:.3f} (0.0%)")

    # Plot comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Left: Top 20 overall features
    top_20_all = feature_df.head(20)
    colors = ['steelblue' if t == 'numerical' else 'coral' for t in top_20_all['type']]
    ax1.barh(range(len(top_20_all)), top_20_all['importance'], color=colors)
    ax1.set_yticks(range(len(top_20_all)))
    ax1.set_yticklabels(top_20_all['feature'], fontsize=9)
    ax1.set_xlabel('Importance')
    ax1.set_title(f'Top 20 Most Important Features (All Types)')
    ax1.invert_yaxis()
    ax1.legend([plt.Rectangle((0,0),1,1, fc='steelblue'), 
                plt.Rectangle((0,0),1,1, fc='coral')], 
               ['Numerical', 'TF-IDF'], loc='lower right')

    # Right: Feature type contribution pie chart
    ax2.pie([numerical_contrib, tfidf_contrib], 
            labels=['Numerical', 'TF-IDF'],
            autopct='%1.1f%%',
            colors=['steelblue', 'coral'],
            startangle=90)
    ax2.set_title('Feature Importance Contribution by Type')

    plt.tight_layout()
    plt.show()

else:
    print(f"⚠️ {best_model_name} does not support feature importance analysis")

## Step 10.5: Analyze TF-IDF Feature Contribution

Check which text features are most important and compare their contribution vs numerical features.

## Step 11: Test Set Evaluation

In [ ]:
# Evaluate best model on test set (using TF-IDF features)
y_test_pred = best_model.predict(X_test_with_text)

# Calculate regression metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

print(f"🎯 {best_model_name} Performance on Test Set:")
print(f"  MAE:  {test_mae:.4f}")
print(f"  MSE:  {test_mse:.4f}")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  R²:   {test_r2:.4f}")

# Error analysis
test_residuals = y_test - y_test_pred
print(f"\n📈 Test Set Error Distribution:")
print(f"  Mean residual: {test_residuals.mean():.4f}")
print(f"  Std residual:  {test_residuals.std():.4f}")
print(f"  Min residual:  {test_residuals.min():.4f}")
print(f"  Max residual:  {test_residuals.max():.4f}")

In [ ]:
# Test set visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Predicted vs Actual
axes[0].scatter(y_test, y_test_pred, alpha=0.5, s=20, color='green')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Engagement Score')
axes[0].set_ylabel('Predicted Engagement Score')
axes[0].set_title(f'Test Set: Predicted vs Actual - {best_model_name}')
axes[0].legend()
axes[0].grid(alpha=0.3)

plt.show()

# Right: Residual plot

test_residuals = y_test - y_test_pred

axes[1].scatter(y_test_pred, test_residuals, alpha=0.5, s=20, color='green')
axes[1].grid(alpha=0.3)

axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Engagement Score')
axes[1].set_ylabel('Residuals (Actual - Predicted)')
axes[1].set_title(f'Test Set: Residual Plot - {best_model_name}')

plt.tight_layout()

plt.show()

## Step 12: Save Models and Results

In [ ]:
import joblib

# Create models directory
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Save best model
model_path = models_dir / f"{best_model_name.lower().replace(' ', '_')}_baseline.pkl"
joblib.dump(best_model, model_path)
print(f"💾 Saved best model: {model_path}")

# Save scaler
scaler_path = models_dir / "scaler_baseline.pkl"
joblib.dump(scaler, scaler_path)
print(f"💾 Saved scaler: {scaler_path}")

# Save TF-IDF vectorizer (NEW!)
tfidf_path = models_dir / "tfidf_vectorizer.pkl"
joblib.dump(tfidf, tfidf_path)
print(f"💾 Saved TF-IDF vectorizer: {tfidf_path}")

# Save results
results_df = comparison_df.copy()
results_df.to_csv(modeling_dir / "baseline_results.csv", index=False)
print(f"💾 Saved results: {modeling_dir / 'baseline_results.csv'}")

# Save test performance
test_results = {
    'model': best_model_name,
    'task': 'regression',
    'target': 'engagement_score',
    'test_mae': test_mae,
    'test_mse': test_mse,
    'test_rmse': test_rmse,
    'test_r2': test_r2,
    'evaluated_at': datetime.now().isoformat(),
    'features': {
        'numerical': len(encoded_feature_cols),
        'tfidf': X_train_tfidf.shape[1],
        'arabert': X_train_bert_reduced.shape[1],
        'total': X_train_with_text.shape[1],
        'strategy': 'combined_all'  # TF-IDF + AraBERT + Numerical
    }
}

with open(modeling_dir / "test_results.json", 'w') as f:
    json.dump(test_results, f, indent=2)
print(f"💾 Saved test results: {modeling_dir / 'test_results.json'}")

print("\n✅ All artifacts saved!")

## Summary and Next Steps

### Key Findings
- **Best Model**: Check cell above for the winning model
- **Validation R² Score**: See comparison table
- **Test R² Score**: See test evaluation
- **Task Type**: Regression (predicting continuous engagement_score)

### Data Splits Saved
All splits have been saved to `data/modeling/`:
- ✅ `train.csv` - Training data
- ✅ `val.csv` - Validation data
- ✅ `test.csv` - Test data
- ✅ `splits_mapping.csv` - Channel to split assignment
- ✅ `split_metadata.json` - Split metadata and configuration

### Next Steps
1. **Hyperparameter Tuning**: Use GridSearchCV or RandomizedSearchCV on best model
3. **Ensemble Methods**: Try stacking or voting regressors
4. **Error Analysis**: Investigate videos with high prediction errors
5. **Cross-Validation**: Implement k-fold CV to validate results
6. **Outlier Analysis**: Examine videos with extreme residuals

### Notes
- Group-based splitting by `channel_id` prevents data leakage
- All videos from the same channel stay in the same split
- Feature scaling applied using StandardScaler

- Models saved in `models/` directory for future use- Models saved in `models/` directory for future use